## Data Extraction with Docling

In this notebook, we'll extract content from PDFs into structured formats:

- **Markdown**: Full document text with page breaks for chunking
- **Images**: Save pages containing large charts/diagrams (>500x500 pixels)
- **Tables**: Extract with 2 paragraphs of context + page number metadata

**Output Structure:**
```
data/rag-data/markdown/{company}/{document}.md
data/rag-data/images/{company}/{document}/page_5.png
data/rag-data/tables/{company}/{document}/table_1_page_5.md
```

https://github.com/docling-project/docling

### 1. Setup and Configuration

In [ ]:
from pathlib import Path
from typing import List, Tuple

from docling_core.types.doc import PictureItem
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.document_converter import DocumentConverter, PdfFormatOption, ConversionResult

In [2]:
## Check torch package run with GPU
import torch

print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

torch.version.cuda

2.6.0+cu124
True
NVIDIA GeForce RTX 3060


'12.4'

In [3]:
import traceback

try:
    from transformers import AutoProcessor
except Exception:
    traceback.print_exc()

In [4]:
# Directory paths
DATA_DIR = "data/rag-data/pdfs"
OUTPUT_MD_DIR = "data/rag-data-todo/markdown"
OUTPUT_IMAGES_DIR = "data/rag-data-todo/images"
OUTPUT_TABLES_DIR = "data/rag-data-todo/tables"

### Metadata Extraction

In [5]:
def extract_metadata_from_filename(filename: str):
    """
    Extract metadata from filename.
    
    Expected format: CompanyName DocType [Quarter] Year.pdf
    Examples:
        - Amazon 10-Q Q1 2024.pdf
        - Microsoft 10-K 2023.pdf
    """

    filename = filename.replace('.pdf', '').replace('.md', '')
    parts = filename.split()

    return {
        'company_name': parts[0],
        'doc_type': parts[1],
        'fiscal_quarter': parts[2] if len(parts)==4 else None,
        'fiscal_year': parts[-1]
    }

extract_metadata_from_filename('apple 10-k 2023.pdf')

{'company_name': 'apple',
 'doc_type': '10-k',
 'fiscal_quarter': None,
 'fiscal_year': '2023'}

### Extract Markdown

In [6]:
pdf_file = Path('data\\rag-data\\pdfs\\apple\\apple 8-k q4 2023.pdf')

print(pdf_file.stem, pdf_file.name)

metadata = extract_metadata_from_filename(pdf_file.stem)
metadata

apple 8-k q4 2023 apple 8-k q4 2023.pdf


{'company_name': 'apple',
 'doc_type': '8-k',
 'fiscal_quarter': 'q4',
 'fiscal_year': '2023'}

In [7]:
import psutil
import os

def print_memory():
    process = psutil.Process(os.getpid())
    print(process.memory_info().rss / 1024**3, "GB")

In [ ]:
def convert_pdf_to_docling(pdf_file: Path):

    pipeline_options = PdfPipelineOptions()
    pipeline_options.images_scale = 2
    pipeline_options.generate_picture_images = True
    pipeline_options.generate_page_images = True

    doc_converter = DocumentConverter(
        format_options={
            InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)
        }
    )

    return doc_converter.convert(pdf_file)

In [ ]:

def save_page_images(doc_converter: ConversionResult, images_dir: Path):
    """
    Find and save pages with large images (>500x500 pixels).
    """

    pages_to_save = set()

    for item in doc_converter.document.iterate_items():
        element = item[0]

        if isinstance(element, PictureItem):
            image = element.get_image(doc_converter.document)

            if image.size[0]>500 and image.size[1]>500:
                page_no = element.prov[0].page_no if element.prov else None

                if page_no:
                    pages_to_save.add(page_no)


    # save images
    for page_no in pages_to_save:
        page = doc_converter.document.pages[page_no]

        page.image.pil_image.save(images_dir/ f"page_{page_no}.png", "PNG")


In [9]:
def extract_context_and_table(lines: List[str], table_index: int):
    """
    Extract context and table content at a specific position.
    
    Args:
        lines: All markdown lines
        table_index: Where the table starts
    
    Returns:
        (combined_content, next_line_index)
    """

    table_lines = []
    i = table_index

    while (i < len(lines)) and (lines[i].startswith('|')):
        table_lines.append(lines[i])
        i = i + 1


    # previous 2 lines as table context
    start = max(0, table_index-2)
    context_lines = lines[start: table_index]

    content = '\n'.join(context_lines) + '\n\n' + '\n'.join(table_lines)

    return content, i
    

In [10]:
def extract_tables_with_context(markdown_text: str):
    """
    Find all tables and extract them with context and page numbers.
    
    Returns:
        List of (content, table_name, page_number)
    """

    lines = markdown_text.split('\n')
    lines = [line for line in lines if line.strip()]
    tables = []
    current_page = 1
    table_num = 1
    i = 0

    while(i< len(lines)):
        # track page numbers
        if '<!-- page break -->' in lines[i]:
            current_page = current_page + 1
            i = i + 1
            continue

        # Table detected
        if lines[i].startswith('|') and lines[i].count('|')>1:
            content, next_i = extract_context_and_table(lines, i)

            tables.append((content, f"table_{table_num}", current_page))
            table_num = table_num + 1
            i = next_i

        else:
            i = i + 1


    return tables
    

In [11]:
def save_tables(markdown_text, tables_dir):

    tables = extract_tables_with_context(markdown_text)

    for table_content, table_name, page_num in tables:
        content_with_page = f"**Page:** {page_num}\n\n{table_content}"
                
        (tables_dir/f"{table_name}_page_{page_num}.md").write_text(content_with_page, encoding='utf-8')


In [12]:
def extract_pdf_content(pdf_file):
    metadata = extract_metadata_from_filename(pdf_file.stem)

    company_name = metadata['company_name']

    md_dir = Path(OUTPUT_MD_DIR) / company_name
    images_dir = Path(OUTPUT_IMAGES_DIR) / company_name / pdf_file.stem
    tables_dir = Path(OUTPUT_TABLES_DIR) / company_name / pdf_file.stem

    for dir_path in [md_dir, images_dir, tables_dir]:
        dir_path.mkdir(parents=True, exist_ok=True)

    doc_converter = convert_pdf_to_docling(pdf_file)

    markdown_text = doc_converter.document.export_to_markdown(page_break_placeholder="<!-- page break -->")

    (md_dir / f"{pdf_file.stem}.md").write_text(markdown_text, encoding='utf-8')

    save_page_images(doc_converter, images_dir)

    save_tables(markdown_text, tables_dir)


In [14]:
# pdf_file = Path('data\\rag-data\\pdfs\\apple\\apple 10-k 2023.pdf')

# extract_pdf_content(pdf_file)

data_path = Path(DATA_DIR)


[INFO] 2026-07-22 10:17:27,846 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-07-22 10:17:27,855 [RapidOCR] download_file.py:60: File exists and is valid: D:\coding\ai_learning\laxmimerit\Multi-Agent-Deep-RAG\.venv\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.onnx
[INFO] 2026-07-22 10:17:27,855 [RapidOCR] main.py:63: Using D:\coding\ai_learning\laxmimerit\Multi-Agent-Deep-RAG\.venv\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.onnx
[INFO] 2026-07-22 10:17:27,913 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-07-22 10:17:27,915 [RapidOCR] download_file.py:60: File exists and is valid: D:\coding\ai_learning\laxmimerit\Multi-Agent-Deep-RAG\.venv\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-07-22 10:17:27,916 [RapidOCR] main.py:63: Using D:\coding\ai_learning\laxmimerit\Multi-Agent-Deep-RAG\.venv\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-07-22 10:17:27,957 [Ra

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

In [15]:
pdf_files = data_path.rglob("*.pdf")
for idx, pdf_file in enumerate(pdf_files):
    print(pdf_file)
    extract_pdf_content(pdf_file)

[INFO] 2026-07-22 09:28:07,667 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-07-22 09:28:07,683 [RapidOCR] download_file.py:60: File exists and is valid: D:\coding\ai_learning\laxmimerit\Multi-Agent-Deep-RAG\.venv\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.onnx
[INFO] 2026-07-22 09:28:07,685 [RapidOCR] main.py:63: Using D:\coding\ai_learning\laxmimerit\Multi-Agent-Deep-RAG\.venv\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.onnx
[INFO] 2026-07-22 09:28:07,750 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-07-22 09:28:07,752 [RapidOCR] download_file.py:60: File exists and is valid: D:\coding\ai_learning\laxmimerit\Multi-Agent-Deep-RAG\.venv\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-07-22 09:28:07,752 [RapidOCR] main.py:63: Using D:\coding\ai_learning\laxmimerit\Multi-Agent-Deep-RAG\.venv\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_mobile.onnx


data\rag-data\pdfs\amazon\amazon 10-k 2023.pdf


[INFO] 2026-07-22 09:28:07,794 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-07-22 09:28:07,811 [RapidOCR] download_file.py:60: File exists and is valid: D:\coding\ai_learning\laxmimerit\Multi-Agent-Deep-RAG\.venv\Lib\site-packages\rapidocr\models\PP-OCRv6_rec_small.onnx
[INFO] 2026-07-22 09:28:07,812 [RapidOCR] main.py:63: Using D:\coding\ai_learning\laxmimerit\Multi-Agent-Deep-RAG\.venv\Lib\site-packages\rapidocr\models\PP-OCRv6_rec_small.onnx


Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

[INFO] 2026-07-22 09:28:49,266 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-07-22 09:28:49,275 [RapidOCR] download_file.py:60: File exists and is valid: D:\coding\ai_learning\laxmimerit\Multi-Agent-Deep-RAG\.venv\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.onnx
[INFO] 2026-07-22 09:28:49,276 [RapidOCR] main.py:63: Using D:\coding\ai_learning\laxmimerit\Multi-Agent-Deep-RAG\.venv\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.onnx
[INFO] 2026-07-22 09:28:49,339 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-07-22 09:28:49,341 [RapidOCR] download_file.py:60: File exists and is valid: D:\coding\ai_learning\laxmimerit\Multi-Agent-Deep-RAG\.venv\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-07-22 09:28:49,342 [RapidOCR] main.py:63: Using D:\coding\ai_learning\laxmimerit\Multi-Agent-Deep-RAG\.venv\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_mobile.onnx


...

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

[INFO] 2026-07-22 09:45:53,655 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-07-22 09:45:53,664 [RapidOCR] download_file.py:60: File exists and is valid: D:\coding\ai_learning\laxmimerit\Multi-Agent-Deep-RAG\.venv\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.onnx
[INFO] 2026-07-22 09:45:53,665 [RapidOCR] main.py:63: Using D:\coding\ai_learning\laxmimerit\Multi-Agent-Deep-RAG\.venv\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.onnx
[INFO] 2026-07-22 09:45:53,729 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-07-22 09:45:53,731 [RapidOCR] download_file.py:60: File exists and is valid: D:\coding\ai_learning\laxmimerit\Multi-Agent-Deep-RAG\.venv\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-07-22 09:45:53,731 [RapidOCR] main.py:63: Using D:\coding\ai_learning\laxmimerit\Multi-Agent-Deep-RAG\.venv\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_mobile.onnx


data\rag-data\pdfs\meta\meta 10-q q4 2024.pdf


[INFO] 2026-07-22 09:45:53,784 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-07-22 09:45:53,803 [RapidOCR] download_file.py:60: File exists and is valid: D:\coding\ai_learning\laxmimerit\Multi-Agent-Deep-RAG\.venv\Lib\site-packages\rapidocr\models\PP-OCRv6_rec_small.onnx
[INFO] 2026-07-22 09:45:53,805 [RapidOCR] main.py:63: Using D:\coding\ai_learning\laxmimerit\Multi-Agent-Deep-RAG\.venv\Lib\site-packages\rapidocr\models\PP-OCRv6_rec_small.onnx


Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]